<div dir="rtl">
<h1>Temperature، بدون تغییر وزن</h1>
<p>درس 62 از 76 · حریصانه‌بودن و Temperature چه فرقی دارند؟ · <code dir="ltr">55-temperature</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-02/55-temperature.html">📖 بازگشت به همین درس</a></p>
<p>توزیع انتخاب را تغییر دهید و ثابت‌ماندن رتبه‌ها و وزن‌های مدل را بررسی کنید.</p><p>پیش‌نیاز: Softmax و تفاوت Logits با احتمال.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>اگر همهٔ Logits را در یک عدد مثبت تقسیم کنیم، آیا شناسهٔ بزرگ‌ترین امتیاز عوض می‌شود؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import torch
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
from mini_gpt.sampling import sampling_distribution
torch.set_num_threads(1)
torch.manual_seed(7)
model = MiniGPT(ModelConfig(8,4,8,2,1,0.0)).eval()
prompt = torch.tensor([[1,2,3]])
with torch.no_grad():
    actual_logits = model(prompt)[0][:,-1,:]
print('real model probabilities:',sampling_distribution(actual_logits).tolist())

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>temperature_probs(Logits, Temperature) برای Logits دوبعدی متناهی و Temperature مثبت، Softmax امتیازهای تقسیم‌شده بر Temperature را در محور آخر برگرداند. برای Temperature صفر یا منفی ValueError بدهید.</p>
</div>

In [ ]:
def temperature_probs(logits, temperature):
    # TODO: تغییر احتمال‌ها، نه وزن‌ها
    return None

In [ ]:
def test_exercise():
    logits = torch.tensor([[2.0,1.0,0.0],[0.0,0.0,0.0]])
    result = temperature_probs(logits,0.5)
    if result is None:
        return False
    torch.testing.assert_close(result,sampling_distribution(logits,temperature=0.5))
    torch.testing.assert_close(result.sum(-1),torch.ones(2))
    torch.testing.assert_close(temperature_probs(logits+9,2.0),temperature_probs(logits,2.0))
    torch.testing.assert_close(temperature_probs(actual_logits,1.3),sampling_distribution(actual_logits,temperature=1.3))
    for invalid in (0.0,-1.0):
        try:
            temperature_probs(logits,invalid)
        except ValueError:
            pass
        else:
            raise AssertionError('reject nonpositive temperature')
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: temperature_probs')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط Temperature را عوض کنید؛ مدل، Prompt و Seed ثابت‌اند. متن مدل آموزش‌ندیده معیار کیفیت نیست؛ تغییر احتمال و ادامه را ببینید.</p>
</div>

In [ ]:
before = {name:value.clone() for name,value in model.state_dict().items()}
for temperature in (0.5,1.0,2.0):
    torch.manual_seed(19)
    print(temperature,model.generate(prompt,6,temperature=temperature).tolist())
assert all(torch.equal(before[name],value) for name,value in model.state_dict().items())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>Temperature صفر روش پیاده‌سازی Greedy نیست. greedy_ids(Logits) را بدون تقسیم بنویسید؛ خروجی باید (B,1) باشد و در تساوی نخستین شناسه را انتخاب کند.</p>
</div>

In [ ]:
try:
    model.generate(prompt,1,temperature=0)
except ValueError as error:
    print('expected failure:',error)
else:
    raise AssertionError('zero temperature must be rejected')

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def greedy_ids(logits):
    # TODO: انتخاب مستقیم بیشترین امتیاز
    return None

In [ ]:
def test_repair():
    result = greedy_ids(torch.tensor([[2.0,1.0],[0.0,3.0]]))
    if result is None:
        return False
    assert result.tolist()==[[0],[1]]
    assert greedy_ids(torch.zeros(1,4)).tolist()==[[0]]
    assert result.dtype==torch.long
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: greedy_ids')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>sampling_distribution و generate همان توابع پروژه‌اند. تغییر Temperature مرحلهٔ تولید را عوض می‌کند و هیچ آموزش تازه‌ای انجام نمی‌دهد.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>چرا تغییر Seed یا Temperature نمی‌تواند اطلاعاتی را که Tokenizer حذف کرده برگرداند؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-02/55-temperature.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/55-temperature.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>